# AdS4 Multi-Minima Landscape (v13)

This notebook visualizes the inverse-problem phase transition identified in v12.

Idea:
- choose a local point on the learned geometry
- perturb the latent geometry parameters `(f, g)` around that point
- evaluate the local probe-loss landscape
- compare **below**, **near**, and **above** the empirical threshold `c* ≈ 0.16`

Goal:
- show that the transition is a **global multi-solution / branching** effect,
  not a local conditioning failure


In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


In [ ]:
# Ground-truth AdS4-style toy background
r = torch.linspace(1.05, 3.5, 240, device=device).view(-1, 1)

def f_true(r):
    return r**2 + 1 - 1/r

def g_true(r):
    return 1/(f_true(r) + 1e-6)

def ee_from_fg(f, g):
    return torch.log(1.0 + 0.6 * f)

def wl_from_fg(f, g):
    return torch.sqrt(f + 1.8 * g + 1e-6)

def geo_from_fg(f, g):
    return torch.sqrt(f + 0.7 * g + 1e-6)

ee_obs = ee_from_fg(f_true(r), g_true(r)).detach()
wl_obs = wl_from_fg(f_true(r), g_true(r)).detach()
geo_obs = geo_from_fg(f_true(r), g_true(r)).detach()


In [ ]:
def structured_geo_target(r, corruption_strength=0.0):
    base = geo_from_fg(f_true(r), g_true(r))
    if corruption_strength == 0.0:
        return base.detach()
    radial_bias = 1.0 + corruption_strength * (0.30 * (r - r.min()) / (r.max() - r.min()))
    oscillation = 1.0 + corruption_strength * 0.15 * torch.sin(2.2 * r)
    return (base * radial_bias * oscillation).detach()


In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x)

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = MLP()
        self.g = MLP()
    def forward(self, r):
        f = F.softplus(self.f(r))
        g = F.softplus(self.g(r))
        return f, g


In [ ]:
def train(corruption_strength=0.0, epochs=1200, consistency_weight=0.15):
    m = Model().to(device)
    opt = torch.optim.Adam(m.parameters(), lr=2e-3)
    geo_target = structured_geo_target(r, corruption_strength)

    loss_hist = []
    for _ in range(epochs):
        opt.zero_grad()
        f, g = m(r)
        ee_pred = ee_from_fg(f, g)
        wl_pred = wl_from_fg(f, g)
        geo_pred = geo_from_fg(f, g)

        loss_ee = ((ee_pred - ee_obs)**2).mean()
        loss_wl = ((wl_pred - wl_obs)**2).mean()
        loss_geo = ((geo_pred - geo_target)**2).mean()
        loss_consistency = ((f * g - 1.0)**2).mean()
        loss = loss_ee + loss_wl + loss_geo + consistency_weight * loss_consistency
        loss.backward()
        opt.step()
        loss_hist.append(loss.item())

    with torch.no_grad():
        f_pred, g_pred = m(r)
        metric_err = ((f_pred - f_true(r)).abs() + (g_pred - g_true(r)).abs()) / 2.0
    return {
        'f_pred': f_pred,
        'g_pred': g_pred,
        'loss_hist': loss_hist,
        'metric_err_mean': metric_err.mean().item(),
        'metric_err_max': metric_err.max().item(),
    }


In [ ]:
# Use representative corruption levels from earlier notebooks
levels = [0.00, 0.16, 0.30]
runs = {c: train(corruption_strength=c) for c in levels}
for c, out in runs.items():
    print(c, out['metric_err_mean'], out['metric_err_max'])


In [ ]:
# Local loss landscape around the midpoint of the learned solution
mid_idx = len(r) // 2

def local_loss_landscape(corruption_strength, f0, g0, span_f=1.2, span_g=0.25, n=120):
    geo_target_mid = structured_geo_target(r, corruption_strength)[mid_idx].item()
    ee_mid = ee_obs[mid_idx].item()
    wl_mid = wl_obs[mid_idx].item()

    f_vals = np.linspace(max(1e-4, f0 - span_f), f0 + span_f, n)
    g_vals = np.linspace(max(1e-4, g0 - span_g), g0 + span_g, n)
    FF, GG = np.meshgrid(f_vals, g_vals)

    ee = np.log(1.0 + 0.6 * FF)
    wl = np.sqrt(np.maximum(FF + 1.8 * GG, 1e-8))
    geo = np.sqrt(np.maximum(FF + 0.7 * GG, 1e-8))
    consistency = (FF * GG - 1.0)**2

    L = (ee - ee_mid)**2 + (wl - wl_mid)**2 + (geo - geo_target_mid)**2 + 0.15 * consistency
    return FF, GG, L


In [ ]:
landscapes = {}
for c in levels:
    f0 = runs[c]['f_pred'][mid_idx].item()
    g0 = runs[c]['g_pred'][mid_idx].item()
    landscapes[c] = local_loss_landscape(c, f0, g0)


In [ ]:
# Main figure: heatmaps + contours
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for col, c in enumerate(levels):
    FF, GG, L = landscapes[c]
    ax = axes[0, col]
    im = ax.imshow(L, origin='lower', aspect='auto',
                   extent=[FF.min(), FF.max(), GG.min(), GG.max()])
    ax.set_title(f'loss heatmap @ c={c:.2f}')
    ax.set_xlabel('f')
    ax.set_ylabel('g')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    ax2 = axes[1, col]
    cs = ax2.contour(FF, GG, L, levels=20)
    ax2.clabel(cs, inline=True, fontsize=7)
    ax2.set_title(f'contours @ c={c:.2f}')
    ax2.set_xlabel('f')
    ax2.set_ylabel('g')

plt.suptitle('AdS4 Multi-Minima Landscape (v13)')
plt.tight_layout()
plt.savefig('/mnt/data/ads4_phase_lock_v13_heatmaps.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: /mnt/data/ads4_phase_lock_v13_heatmaps.png')


In [ ]:
# Reconstruction examples for the same three corruption levels
plt.figure(figsize=(12, 4))
for i, c in enumerate(levels, start=1):
    row = runs[c]
    plt.subplot(1, 3, i)
    plt.plot(r.cpu(), f_true(r).cpu(), label='f true')
    plt.plot(r.cpu(), row['f_pred'].cpu(), '--', label=f'f @{c:.2f}')
    plt.plot(r.cpu(), g_true(r).cpu(), label='g true')
    plt.plot(r.cpu(), row['g_pred'].cpu(), ':', label=f'g @{c:.2f}')
    plt.title(f'corruption={c:.2f}')
    plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig('/mnt/data/ads4_phase_lock_v13_recon.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: /mnt/data/ads4_phase_lock_v13_recon.png')
